<a href="https://colab.research.google.com/github/hOshi123456/Proyecto_Inteligencia_Artificial_MarianRomero/blob/main/Proyecto_IA_Ventas_Mistral_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Instalar dependencias
!pip install -q pandas numpy python-dotenv requests

In [ ]:
import pandas as pd
import numpy as np
import os
import requests

print("Entorno preparado correctamente ✅")

Entorno preparado correctamente ✅


In [ ]:
#Adjuntar archivo del dataset descargado
from google.colab import files

uploaded = files.upload()

Saving sales_data_sample.csv to sales_data_sample.csv


In [ ]:
#Ver el nomnbre del archivo que se subió
import os

os.listdir()

['.config', 'sales_data_sample.csv', 'sample_data']

In [ ]:
#Cargar el dataset de ventas#
archivo = "sales_data_sample.csv"

df = pd.read_csv(archivo, encoding="latin1")

print("Dataset cargado correctamente ✅")
print("Filas y columnas:", df.shape)

Dataset cargado correctamente ✅
Filas y columnas: (2823, 25)


In [ ]:
# Ver las primeras 5 filas del dataset

df.head()

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


In [ ]:
# PASO 4: Explorar el dataset

print("Columnas del dataset:")
print(df.columns.tolist())

print("\nInformación general:")
df.info()

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nCantidad de filas duplicadas:")
print(df.duplicated().sum())

Columnas del dataset:
['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']

Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2823 entries, 0 to 2822
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDERNUMBER       2823 non-null   int64  
 1   QUANTITYORDERED   2823 non-null   int64  
 2   PRICEEACH         2823 non-null   float64
 3   ORDERLINENUMBER   2823 non-null   int64  
 4   SALES             2823 non-null   float64
 5   ORDERDATE         2823 non-null   object 
 6   STATUS            2823 non-null   object 
 7   QTR_ID            2823 non-null   int64  
 8   MONTH_ID          2823 non-null   int64  

In [ ]:
#Normalización y limpieza básica del dataset

# Creamos una copia para no dañar el dataset original
df_limpio = df.copy()

# Normalizar nombres de columnas: minúsculas y sin espacios
df_limpio.columns = (
    df_limpio.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

# Quitar espacios extra en columnas de texto
columnas_texto = df_limpio.select_dtypes(include=["object"]).columns

for columna in columnas_texto:
    df_limpio[columna] = df_limpio[columna].astype(str).str.strip()

# Convertir orderdate a formato fecha
df_limpio["orderdate"] = pd.to_datetime(df_limpio["orderdate"], errors="coerce")

# Rellenar valores nulos en columnas específicas
df_limpio["addressline2"] = df_limpio["addressline2"].replace("nan", "No especificado")
df_limpio["state"] = df_limpio["state"].replace("nan", "No especificado")
df_limpio["postalcode"] = df_limpio["postalcode"].replace("nan", "No especificado")
df_limpio["territory"] = df_limpio["territory"].replace("nan", "No especificado")

# Verificar resultado
print("Dataset normalizado correctamente ✅")
print("Filas y columnas:", df_limpio.shape)

print("\nValores nulos restantes:")
print(df_limpio.isnull().sum())

df_limpio.head()

Dataset normalizado correctamente ✅
Filas y columnas: (2823, 25)

Valores nulos restantes:
ordernumber         0
quantityordered     0
priceeach           0
orderlinenumber     0
sales               0
orderdate           0
status              0
qtr_id              0
month_id            0
year_id             0
productline         0
msrp                0
productcode         0
customername        0
phone               0
addressline1        0
addressline2        0
city                0
state               0
postalcode          0
country             0
territory           0
contactlastname     0
contactfirstname    0
dealsize            0
dtype: int64


,ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,qtr_id,month_id,year_id,...,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,1,2,2003,...,897 Long Airport Avenue,No especificado,NYC,NY,10022,USA,No especificado,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,2,5,2003,...,59 rue de l'Abbaye,No especificado,Reims,No especificado,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,No especificado,Paris,No especificado,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,2003-08-25,Shipped,3,8,2003,...,78934 Hillside Dr.,No especificado,Pasadena,CA,90003,USA,No especificado,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,2003-10-10,Shipped,4,10,2003,...,7734 Strong St.,No especificado,San Francisco,CA,No especificado,USA,No especificado,Brown,Julie,Medium


In [ ]:
#Exportar el dataset limpio

nombre_archivo_limpio = "ventas_limpio.csv"

df_limpio.to_csv(nombre_archivo_limpio, index=False, encoding="utf-8")

print("Archivo exportado correctamente ✅")
print("Nombre del archivo:", nombre_archivo_limpio)

Archivo exportado correctamente ✅
Nombre del archivo: ventas_limpio.csv


In [ ]:
from google.colab import files

files.download("ventas_limpio.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Probar conexión con Mistral usando API Key protegida

from google.colab import userdata
import requests

MistralApi = userdata.get("MistralApi")

url = "https://api.mistral.ai/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {MistralApi}",
    "Content-Type": "application/json"
}

data = {
    "model": "mistral-small-latest",
    "messages": [
        {
            "role": "user",
            "content": "Hola, responde en español: ¿qué es un dataset de ventas?"
        }
    ],
    "temperature": 0.3
}

response = requests.post(url, headers=headers, json=data)

print("Código de estado:", response.status_code)

if response.status_code == 200:
    respuesta = response.json()
    print("\nRespuesta de Mistral:")
    print(respuesta["choices"][0]["message"]["content"])
else:
    print("\nError:")
    print(response.text)

Código de estado: 200

Respuesta de Mistral:
¡Hola! Un **dataset de ventas** es un conjunto de datos estructurados que contiene información relacionada con las transacciones comerciales de una empresa o negocio. Estos datos suelen incluir detalles como:

- **Productos o servicios vendidos** (nombres, categorías, precios).
- **Clientes** (identificadores, datos demográficos, historial de compras).
- **Fechas y horarios** de las transacciones.
- **Cantidades vendidas** y montos totales.
- **Métodos de pago** (efectivo, tarjeta, transferencia, etc.).
- **Ubicaciones** (tiendas físicas, online, regiones geográficas).
- **Descuentos o promociones** aplicadas.

### **Ejemplo de estructura de un dataset de ventas (en formato tabla):**
| **ID_Venta** | **Fecha**       | **Producto**  | **Cantidad** | **Precio_Unitario** | **Cliente_ID** | **Tienda** | **Método_Pago** |
|--------------|-----------------|---------------|--------------|---------------------|----------------|------------|---------

In [ ]:
#Crear resumen del dataset limpio para Mistral

resumen_dataset = f"""
Resumen del dataset de ventas:

Cantidad de filas: {df_limpio.shape[0]}
Cantidad de columnas: {df_limpio.shape[1]}

Columnas disponibles:
{', '.join(df_limpio.columns)}

Total de ventas: {df_limpio['sales'].sum():,.2f}

Cantidad total de productos vendidos:
{df_limpio['quantityordered'].sum()}

Cantidad de órdenes únicas:
{df_limpio['ordernumber'].nunique()}

Países presentes en el dataset:
{', '.join(df_limpio['country'].dropna().unique())}

Líneas de productos:
{', '.join(df_limpio['productline'].dropna().unique())}

Estados de las órdenes:
{', '.join(df_limpio['status'].dropna().unique())}

Ventas por línea de producto:
{df_limpio.groupby('productline')['sales'].sum().sort_values(ascending=False).to_string()}

Ventas por país:
{df_limpio.groupby('country')['sales'].sum().sort_values(ascending=False).head(10).to_string()}

Ventas por tamaño de trato:
{df_limpio.groupby('dealsize')['sales'].sum().sort_values(ascending=False).to_string()}
"""

print(resumen_dataset)


Resumen del dataset de ventas:

Cantidad de filas: 2823
Cantidad de columnas: 25

Columnas disponibles:
ordernumber, quantityordered, priceeach, orderlinenumber, sales, orderdate, status, qtr_id, month_id, year_id, productline, msrp, productcode, customername, phone, addressline1, addressline2, city, state, postalcode, country, territory, contactlastname, contactfirstname, dealsize

Total de ventas: 10,032,628.85

Cantidad total de productos vendidos:
99067

Cantidad de órdenes únicas:
307

Países presentes en el dataset:
USA, France, Norway, Australia, Finland, Austria, UK, Spain, Sweden, Singapore, Canada, Japan, Italy, Denmark, Belgium, Philippines, Germany, Switzerland, Ireland

Líneas de productos:
Motorcycles, Classic Cars, Trucks and Buses, Vintage Cars, Planes, Ships, Trains

Estados de las órdenes:
Shipped, Disputed, In Process, Cancelled, On Hold, Resolved

Ventas por línea de producto:
productline
Classic Cars        3919615.66
Vintage Cars        1903150.84
Motorcycles    

In [ ]:
#Función para preguntar a Mistral sobre el dataset

from google.colab import userdata
import requests

MistralApi = userdata.get("MistralApi")

url = "https://api.mistral.ai/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {MistralApi}",
    "Content-Type": "application/json"
}

def preguntar_mistral(pregunta):
    prompt = f"""
Eres un asistente de inteligencia artificial especializado en análisis de datos.

Debes responder únicamente usando la información del siguiente resumen del dataset de ventas.

{resumen_dataset}

Pregunta del usuario:
{pregunta}

Responde en español, de forma clara y breve.
"""

    data = {
        "model": "mistral-small-latest",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.3
    }

    response = requests.post(url, headers=headers, json=data)

    if response.status_code == 200:
        respuesta = response.json()
        return respuesta["choices"][0]["message"]["content"]
    else:
        return f"Error {response.status_code}: {response.text}"

In [ ]:
pregunta = "¿Cuál es la línea de producto con más ventas?"
respuesta = preguntar_mistral(pregunta)

print(respuesta)

La línea de producto con más ventas es **Classic Cars**, con un total de **3,919,615.66**.
